In [0]:
%pip install pymupdf
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "datasheet_project")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

In [0]:
volume_path = "/Volumes/datasheet_project/bronze/raw_pdfs"

files = dbutils.fs.ls(volume_path)
for f in files:
    print(f.name, "-", f.size, "bytes")

In [0]:
import fitz  # PyMuPDF

test_file = "/Volumes/datasheet_project/bronze/raw_pdfs/atmega328pb.pdf"

doc = fitz.open(test_file)
text = ""
for page in doc:
    text += page.get_text()

print(f"Pages: {len(doc)}")
print(f"Characters: {len(text)}")
print("--- First 500 chars ---")
print(text[:500])

In [0]:
import pymupdf
from pyspark.sql import Row
from datetime import datetime

volume_path = "/Volumes/datasheet_project/bronze/raw_pdfs"
files = dbutils.fs.ls(volume_path)

rows = []
for f in files:
    if not f.name.lower().endswith('.pdf'):
        continue
    doc = pymupdf.open(f.path.replace("dbfs:", ""))
    text = ""
    for page in doc:
        text += page.get_text()
    
    rows.append(Row(
        file_name=f.name,
        file_size_bytes=f.size,
        num_pages=len(doc),
        raw_text=text,
        ingested_at=datetime.now()
    ))
    doc.close()

df = spark.createDataFrame(rows)
df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.datasheets_raw")

print(f"✅ Saved {df.count()} rows to bronze.datasheets_raw")

In [0]:
df = spark.table(f"{catalog}.bronze.datasheets_raw")
df.select("file_name", "file_size_bytes", "num_pages", "ingested_at").show(truncate=False)
print(f"Total rows: {df.count()}")